# LeetCode 496: Next Greater Element I

**Difficulty**: Easy  
**Topics**: Array, Hash Table, Stack, Monotonic Stack  
**Link**: [LeetCode Problem](https://leetcode.com/problems/next-greater-element-i/)

---

## Problem Statement

The **next greater element** of some element `x` in an array is the **first greater** element that is **to the right** of `x` in the same array.

You are given two **distinct** 0-indexed integer arrays `nums1` and `nums2`, where `nums1` is a subset of `nums2`.

For each `0 <= i < nums1.length`, find the index `j` such that `nums1[i] == nums2[j]` and determine the **next greater element** of `nums2[j]` in `nums2`. If there is no next greater element, then the answer for this query is `-1`.

Return an array `ans` of length `nums1.length` such that `ans[i]` is the **next greater element** as described above.

### Constraints

- `1 <= nums1.length <= nums2.length <= 1000`
- `0 <= nums1[i], nums2[i] <= 10^4`
- All integers in `nums1` and `nums2` are **unique**
- All the integers of `nums1` also appear in `nums2`

### Examples

**Example 1:**
```
Input: nums1 = [4,1,2], nums2 = [1,3,4,2]
Output: [-1,3,-1]
Explanation:
- For number 4 in nums1: no next greater in nums2 → -1
- For number 1 in nums1: next greater in nums2 is 3
- For number 2 in nums1: no next greater in nums2 → -1
```

**Example 2:**
```
Input: nums1 = [2,4], nums2 = [1,2,3,4]
Output: [3,-1]
Explanation:
- For number 2 in nums1: next greater in nums2 is 3
- For number 4 in nums1: no next greater in nums2 → -1
```

---

## Approach 1: Brute Force

### Intuition

For each element in `nums1`:
1. Find its position in `nums2`
2. Scan to the right in `nums2` to find the first greater element
3. If found, record it; otherwise, record -1

### Algorithm

```
For each num in nums1:
    Find index of num in nums2
    Scan from that index to the right
    Find first element > num
    If found, add to result
    Else add -1
```

### Complexity

- **Time**: O(n × m) where n = len(nums1), m = len(nums2)
  - For each element in nums1, we might scan all of nums2
- **Space**: O(1) excluding output array

In [ ]:
def nextGreaterElement_bruteforce(nums1, nums2):
    """
    Brute force approach.
    Time: O(n × m), Space: O(1)
    """
    result = []
    
    for num in nums1:
        # Find position of num in nums2
        idx = nums2.index(num)
        
        # Scan to the right for next greater
        next_greater = -1
        for j in range(idx + 1, len(nums2)):
            if nums2[j] > num:
                next_greater = nums2[j]
                break
        
        result.append(next_greater)
    
    return result

# Test
print(nextGreaterElement_bruteforce([4,1,2], [1,3,4,2]))  # [-1, 3, -1]
print(nextGreaterElement_bruteforce([2,4], [1,2,3,4]))    # [3, -1]

---

## Approach 2: Monotonic Stack + Hash Map (Optimal)

### Intuition

The key insight is to use a **monotonic stack** to efficiently find the next greater element for ALL elements in `nums2` in one pass.

**Why this works:**
1. We process `nums2` from left to right
2. We maintain a decreasing stack (values decrease from bottom to top)
3. When we see a larger element, it's the "next greater" for all smaller elements in the stack
4. We store these mappings in a hash map: `{element: next_greater}`
5. Finally, we lookup each element in `nums1` in the hash map

### Visual Example

For `nums2 = [1, 3, 4, 2]`:

```
Step 0: See 1
  Stack: [1]
  Map: {}

Step 1: See 3
  3 > 1, so 1's next greater is 3
  Pop 1, add to map: {1: 3}
  Stack: [3]

Step 2: See 4
  4 > 3, so 3's next greater is 4
  Pop 3, add to map: {1: 3, 3: 4}
  Stack: [4]

Step 3: See 2
  2 < 4, no pops
  Stack: [4, 2]
  Map: {1: 3, 3: 4}

Elements still in stack (4, 2) have no next greater → -1
```

### Algorithm

```
1. Create empty stack and hash map
2. For each num in nums2:
     While stack not empty AND stack.top < num:
         map[stack.pop()] = num
     stack.push(num)
3. For each num in nums1:
     result.append(map.get(num, -1))
```

### Complexity

- **Time**: O(n + m) where n = len(nums1), m = len(nums2)
  - O(m) to build the next greater map (each element pushed/popped once)
  - O(n) to lookup results for nums1
- **Space**: O(m) for stack and hash map

In [ ]:
def nextGreaterElement(nums1, nums2):
    """
    Optimal solution using monotonic stack.
    Time: O(n + m), Space: O(m)
    """
    # Build next greater map for all elements in nums2
    next_greater = {}
    stack = []
    
    for num in nums2:
        # Current num is the next greater for all smaller elements in stack
        while stack and stack[-1] < num:
            next_greater[stack.pop()] = num
        stack.append(num)
    
    # Lookup results for nums1
    return [next_greater.get(num, -1) for num in nums1]

# Test
print(nextGreaterElement([4,1,2], [1,3,4,2]))  # [-1, 3, -1]
print(nextGreaterElement([2,4], [1,2,3,4]))    # [3, -1]

### Detailed Walkthrough

Let's trace through `nums1 = [4,1,2], nums2 = [1,3,4,2]`:

In [ ]:
def nextGreaterElement_verbose(nums1, nums2):
    """Verbose version showing each step."""
    next_greater = {}
    stack = []
    
    print(f"Building next greater map for nums2 = {nums2}\n")
    
    for i, num in enumerate(nums2):
        print(f"Step {i}: Processing {num}")
        print(f"  Stack before: {stack}")
        
        # Pop smaller elements
        while stack and stack[-1] < num:
            popped = stack.pop()
            next_greater[popped] = num
            print(f"  {popped} < {num}, so next_greater[{popped}] = {num}")
        
        stack.append(num)
        print(f"  Stack after: {stack}")
        print(f"  Map: {next_greater}\n")
    
    print(f"Final map: {next_greater}")
    print(f"Elements still in stack {stack} have no next greater\n")
    
    # Lookup for nums1
    result = []
    print(f"Looking up results for nums1 = {nums1}:")
    for num in nums1:
        ans = next_greater.get(num, -1)
        result.append(ans)
        print(f"  {num} → {ans}")
    
    print(f"\nFinal result: {result}")
    return result

nextGreaterElement_verbose([4,1,2], [1,3,4,2])

---

## Edge Cases

In [ ]:
# Edge case 1: Single element
print("Single element:")
print(nextGreaterElement([1], [1]))  # [-1]
print()

# Edge case 2: nums1 = nums2
print("nums1 = nums2:")
print(nextGreaterElement([1,2,3], [1,2,3]))  # [2, 3, -1]
print()

# Edge case 3: Decreasing sequence
print("Decreasing sequence:")
print(nextGreaterElement([5,4,3], [5,4,3,2,1]))  # [-1, -1, -1]
print()

# Edge case 4: Increasing sequence
print("Increasing sequence:")
print(nextGreaterElement([1,2,3], [1,2,3,4,5]))  # [2, 3, 4]
print()

# Edge case 5: All elements in nums1 have no next greater
print("No next greater:")
print(nextGreaterElement([5,4], [1,2,3,4,5]))  # [-1, -1]

---

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity | Notes |
|----------|----------------|------------------|-------|
| Brute Force | O(n × m) | O(1) | Simple but slow for large inputs |
| Monotonic Stack | O(n + m) | O(m) | Optimal, uses extra space for map |

Where n = len(nums1), m = len(nums2)

### When to Use Each

- **Brute Force**: Only for very small inputs or when space is extremely limited
- **Monotonic Stack**: Preferred solution for interviews and production code

---

## Common Mistakes

### Mistake 1: Forgetting to Handle Elements with No Next Greater

**Wrong**:
```python
return [next_greater[num] for num in nums1]  # KeyError if num not in map
```

**Right**:
```python
return [next_greater.get(num, -1) for num in nums1]  # Returns -1 if not found
```

### Mistake 2: Using Wrong Comparison in Stack

**Wrong**:
```python
while stack and stack[-1] > num:  # Wrong direction
```

**Right**:
```python
while stack and stack[-1] < num:  # Pop smaller elements
```

### Mistake 3: Processing nums1 Instead of nums2

You must build the next greater map from `nums2`, not `nums1`, because `nums2` contains all the elements and their relationships.

---

## Related Problems

- [503. Next Greater Element II](https://leetcode.com/problems/next-greater-element-ii/) - Circular array version
- [739. Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) - Return distance instead of value
- [556. Next Greater Element III](https://leetcode.com/problems/next-greater-element-iii/) - Number permutation variant
- [1019. Next Greater Node In Linked List](https://leetcode.com/problems/next-greater-node-in-linked-list/) - Linked list version

---

## Key Takeaways

1. **Monotonic stack** is the optimal pattern for "next greater element" problems
2. Build a **hash map** to store next greater relationships
3. Process `nums2` to build the map, then lookup `nums1`
4. Use `dict.get(key, default)` to handle missing keys safely
5. Time complexity improves from O(n×m) to O(n+m)
6. The stack maintains a **decreasing** order for next greater queries
7. Each element is pushed and popped at most once → amortized O(n)